CNN 卷积神经网络：从卷积核到图像分类

卷积神经网络（Convolutional Neural Network，CNN）是一类利用局部连接和参数共享处理网格数据的神经网络。图像是最常见的例子：卷积层从局部区域提取边缘、纹理和形状等特征，后续层逐步组合这些特征，最终完成分类、检测或分割。

本文依次介绍图像张量、卷积核、步长、填充、特征图、池化、感受野，以及使用 PyTorch 搭建和检查一个小型 CNN。


In [1]:
import torch
from torch import nn

In [2]:

images = torch.randn(32, 3, 64, 64)
print(images.shape)  # torch.Size([32, 3, 64, 64])

torch.Size([32, 3, 64, 64])


In [4]:
images = torch.randn(32, 3, 64, 64)
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3)
output_images = conv(images)
print(images.shape)  # [32, 3, 64, 64]
print(output_images.shape) # [32, 16, 64, 64], 16表示输出了16个特征图

torch.Size([32, 3, 64, 64])
torch.Size([32, 16, 62, 62])


In [6]:
## 4. `nn.Conv2d` 与卷积核
in_channels, out_channels, kernel_size = 3, 16, 3
nn.Conv2d(
    in_channels,
    out_channels,
    kernel_size,
    stride=1,
    padding=0,
    dilation=1,
    groups=1,
    bias=True,
)

# | 参数 | 含义 |
# | --- | --- |
# | `in_channels` | 输入通道数，必须与输入 Tensor 的 C 维相同 |
# | `out_channels` | 卷积层输出的特征图（Feature Maps）数量，也是输出特征图的通道数 |
# | `kernel_size` | 核的高和宽；整数表示两边相同，也可传 `(kh, kw)` |
# | `stride` | 核每次移动的步长，整数或 `(sh, sw)` |
# | `padding` | 输入边界补的像素数，整数或 `(ph, pw)`；也支持部分字符串模式 |
# | `dilation` | 核元素之间的间距，用于扩大有效核覆盖范围 |
# | `groups` | 输入、输出通道分组数；影响通道连接方式 |
# | `bias` | 是否为每个输出通道学习偏置 |
#
# 最常见的普通卷积令 `groups=1`，每个输出通道都能组合全部输入通道。若输入是 3 通道、输出是 16 通道、核大小为 3：
# 权重形状是 `[out_channels, in_channels / groups, kh, kw]`；偏置形状是 `[out_channels]`。上例权重形状为 `[16, 3, 3, 3]`。


Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))

In [7]:
## 5. 步长 `stride`
# 步长表示卷积核在输入上移动多少格。`stride=1` 会逐格扫描；`stride=2` 每次移动两格，空间尺寸通常会缩小。横纵步长可以不同：`stride=(2, 1)`。


In [8]:
## 6. 填充 `padding` 与输出尺寸
# 不填充（`padding=0`）时，卷积窗口不能越过图像边界，输出尺寸会变小。填充通常在四周补零，使边缘位置也能参与计算，并控制输出空间大小。

def conv_out_size(length, kernel_size, stride=1, padding=0, dilation=1):
    return (length + 2 * padding - dilation * (kernel_size - 1) - 1) // stride + 1

print(conv_out_size(32, kernel_size=3, stride=2, padding=1))  # 16

16


In [9]:
## 7. 特征图与通道
# 卷积层输出的每个通道是一张特征图（feature map）。`out_channels=16` 意味着产生 16 张特征图，而不是把输入变成 16 张图片；batch 维仍然保留。
conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)
x = torch.randn(4, 3, 28, 28)
y = conv(x)
print(y.shape)  # [4, 16, 28, 28]，批量batch是4，输出特征图数量是16，28是图像长，28是图像宽

torch.Size([4, 16, 28, 28])


In [13]:
# 卷积和池化的区别：

# 在 CNN（卷积神经网络）中，Conv（卷积）和 Pool（池化）是两种不同的操作：
# - Conv（卷积）：负责提取特征，学习图片中有什么。卷积的核心作用是：通过卷积核（Kernel）在图片上滑动，对局部区域进行计算，从而提取特征。卷积核中的权重是自动学习出来的
# - Pool（池化）：负责压缩特征，减少数据量，保留重要信息。池化的核心作用是：对特征图进行下采样（Downsampling），缩小特征图尺寸，减少计算量。
# 可以简单理解为：卷积负责发现特征，池化负责精简特征。
# 池化一般可以分为最大（选择池化窗口内最大的一个值保留）池化和平均（将池化窗口内所有的值求平均）池化，池化也有一个像卷积一个的n*n的窗口

# Conv 和 Pool 的核心区别
# 对比维度	Conv 卷积	Pool 池化
# 核心作用	提取特征	压缩特征
# 是否有可训练参数	有	通常没有
# 是否改变特征图尺寸	可以	通常会缩小
# 核心计算	乘法、加法	最大值或平均值
# 是否学习特征	是	否
# 主要目的	发现图像特征	降低计算量、减少空间冗余
# 这里需要注意：Conv 的输出尺寸不一定缩小，可以通过 stride 和 padding 控制。


In [21]:
## 8. `nn.MaxPool2d` 最大池化
max_pool = nn.MaxPool2d(
    kernel_size=2,
    stride=2
)
# 这个tensor有四维，分别表示批量，特征数，长，宽。所以以下x表示：1个批次1个特征长为6宽为6的图像
"""
1  2  3  4  5 6
2  3  4  5  6 7
3  4  5  6  7 8
4  5  6  7  8 9
5  6  7  8  9 0
6  7  8  9  0 1
"""
x = torch.tensor([[[
    [1, 2, 3, 4, 5, 6],
    [2, 3, 4, 5, 6, 7],
    [3, 4, 5, 6, 7, 8],
    [4, 5, 6, 7, 8, 9],
    [5, 6, 7, 8, 9, 0],
    [6, 7, 8, 9, 0, 1]
]]])
print(max_pool(x))  # 每个 2x2 窗口保留最大值 4

tensor([[[[3, 5, 7],
          [5, 7, 9],
          [7, 9, 9]]]])


In [22]:
# 平均池化
agv_pool = nn.AvgPool2d(kernel_size=2, stride=2)
print(agv_pool(x))

tensor([[[[2, 4, 6],
          [4, 6, 8],
          [6, 8, 2]]]])


In [24]:
## 9. 感受野 `receptive field`
# 某个深层特征图位置所依赖的原始输入区域称为它的感受野。浅层卷积的感受野较小，通常捕捉局部模式；连续堆叠卷积或下采样后，深层位置能整合更大区域的信息。
# 感受野：也就是当前这层特征图中某个像素点点数据来自上层图的哪些像素点，那么上层图的这些像素点就是当前这层图这个像素点的感受野。比如上述最大池化中的[0,0]位置的3来自原始图的[0,0]、[0,1]、[1,0]、[1,1]，那么这四个点就是最大池化中[0,0]位置的感受野。注意：感受野是特征图区域对原始图片输入区域对，所以要通过以上方式一直追溯到原始图片上，原始图片上的那些位置就是这个特征图这个位置对应的感受野

In [27]:
## 10. 组合卷积模块
# 典型 CNN 中 Conv 和 Pool 组合结构：（卷积 -> 归一化 -> 激活 -> 池化）* N -> 展平 -> 线性 -> 概率

"""

输入图片
   |
   v
Conv2d       # 卷积 提取特征
   |
   v
Norm         # 归一化 规范化特征，提高训练稳定性
   |
   v
ReLU         # 激活 增加非线性
   |
   v
MaxPool2d    # 池化 下采样，压缩特征图
   |
   v
Conv2d       # 提取更复杂的特征
   |
   v
Norm
   |
   v
ReLU
   |
   v
MaxPool2d
   |
   v
Flatten      # 展平特征
   |
   v
Linear       # 全连接分类
   |
   v
Softmax      # 输出分类概率（按需使用）
"""

# CNN Block（卷积块）
cnn_block = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1, bias=True),
    nn.BatchNorm2d(num_features=16), # num_features 必须与 out_channels 对应
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
)

x = torch.randn(8, 3, 32, 32)
y = cnn_block(x)
print(y.shape)  # [8, 16, 16, 16], 8是批量，第一个16是out_channels=16，第二、三个16是经过MaxPool的结果，
# Pool 池化之后的特征图长宽，主要由 kernel_size、stride、padding、dilation 四个参数决定。
# 常见的 MaxPool2d(kernel_size=2, stride=2)，它会让特征图的宽度和高度缩小一半
"""
高度：
$
H_{out}=\left\lfloor
\frac{H_{in}+2P-D(K-1)-1}{S}+1
\right\rfloor
$

宽度：
$
W_{out}=\left\lfloor
\frac{W_{in}+2P-D(K-1)-1}{S}+1
\right\rfloor
$
"""


torch.Size([8, 16, 16, 16])



高度：
$
H_{out}=\left\lfloor
\frac{H_{in}+2P-D(K-1)-1}{S}+1
\right\rfloor
$

宽度：
$
W_{out}=\left\lfloor
\frac{W_{in}+2P-D(K-1)-1}{S}+1
\right\rfloor
$



In [33]:
## 10. 组合卷积模块
# 多卷积块组合
# `BatchNorm2d(num_features)` 中 `num_features` 是通道数；它在训练时使用 batch 统计量并维护运行统计量，在 `eval()` 时使用运行统计量。小 batch、分布式或推理场景要考虑其行为。`bias=False` 常与紧随其后的 BatchNorm 搭配，因为归一化层已有可学习偏移；这是常见配置，不是硬性规则。

features = nn.Sequential(
    # Block 1
    nn.Conv2d(3, 16, 3, padding=1, bias=False),
    nn.BatchNorm2d(16), # # 这层的 num_features=16 来自 上面的 out_channels=16
    nn.ReLU(),
    nn.MaxPool2d(2, 2),

    # Block 2
    nn.Conv2d(16, 32, 3, padding=1, bias=False), # 这层的 in_channels=16 来自 上一层的 out_channels=16
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),

    # Block 3
    nn.Conv2d(32, 64, 3, padding=1, bias=False), # 这层的 in_channels=32 来自 上一层的 out_channels=32
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),
)

# 展平层 + 线性层
classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(64 * 4 * 4, 10) # 输入特征是 64*4*4，输出分类数是10
)


input_images = torch.randn(32, 3, 32, 32)
print(f"input_images.shape: {input_images.shape}") # [32, 3, 32, 32]
output_images = features(input_images)
print(f"output_images.shape: {output_images.shape}") # [32, 64, 4, 4], 32来自batch，64来自最终的block3的输出特征图数量，4 = 32 / 2 / 2 / 2
flatten_images = nn.Flatten()(output_images)
print(f"flatten_images.shape: {flatten_images.shape}") # [32, 1024], 32来自batch，最后输出的64张4*4的图片展平成 64 * 4 * 4 = 1024
linear_images = nn.Linear(64 * 4 * 4, 10)(flatten_images)
print(f"linear_images.shape: {linear_images.shape}") # [32, 1024], 32来自batch，10来自线性层分类输出


input_images.shape: torch.Size([32, 3, 32, 32])
output_images.shape: torch.Size([32, 64, 4, 4])
flatten_images.shape: torch.Size([32, 1024])
linear_images.shape: torch.Size([32, 10])


In [40]:
# 将以上转化为一个cnn的类
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SmallCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16), # # 这层的 num_features=16 来自 上面的 out_channels=16
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 2
            nn.Conv2d(16, 32, 3, padding=1, bias=False), # 这层的 in_channels=16 来自 上一层的 out_channels=16
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 3
            nn.Conv2d(32, 64, 3, padding=1, bias=False), # 这层的 in_channels=32 来自 上一层的 out_channels=32
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        # 展平层 + 线性层
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, num_classes) # 输入特征是 64*4*4，输出分类数是 num_classes
        )
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

# 多分类训练通常把 logits 直接传给 `nn.CrossEntropyLoss()`，标签形状为 `[N]`、dtype 为 `torch.long`。训练时不要先对 logits 做 softmax；推理时如需概率，再使用 `torch.softmax(logits, dim=1)`。


in_images = torch.randn(32, 3, 32, 32)
model = SmallCNN()
out_images = model(in_images)
print(out_images.shape)
print()

# 普通卷积的可学习参数量（包含 bias）为：
# out_channels * (in_channels / groups) * kernel_height * kernel_width + (out_channels if bias else 0)
for key, weight in model.named_parameters():
    print(key, weight.shape)





torch.Size([32, 10])

features.0.weight torch.Size([16, 3, 3, 3])
features.1.weight torch.Size([16])
features.1.bias torch.Size([16])
features.4.weight torch.Size([32, 16, 3, 3])
features.5.weight torch.Size([32])
features.5.bias torch.Size([32])
features.8.weight torch.Size([64, 32, 3, 3])
features.9.weight torch.Size([64])
features.9.bias torch.Size([64])
classifier.1.weight torch.Size([10, 1024])
classifier.1.bias torch.Size([10])


In [41]:
## 14. 常见 CNN 层与用途

# | 层 | 作用 | 常见注意点 |
# | --- | --- | --- |
# | `nn.Conv2d` | 学习局部空间特征 | 输入通道、输出 shape、padding 和参数量 |
# | `nn.ReLU` / `nn.GELU` | 引入非线性 | 仅堆叠线性卷积仍等价于线性变换 |
# | `nn.MaxPool2d` | 局部最大值下采样 | 丢失空间细节；默认 stride 通常等于窗口 |
# | `nn.AvgPool2d` | 局部平均下采样 | 平滑局部响应 |
# | `nn.AdaptiveAvgPool2d` | 输出指定空间尺寸 | 常用来适配不同输入大小 |
# | `nn.BatchNorm2d` | 按通道归一化激活 | train/eval 行为不同；小 batch 要评估适用性 |
# | `nn.Dropout2d` | 按通道丢弃特征图 | 仅用于训练正则化，eval 时关闭 |
# | `nn.Flatten` | 展开通道与空间维 | 后接全连接层时核对乘积尺寸 |
# | `nn.Linear` | 汇总特征并输出类别分数 | 多分类输出 logits，不提前 softmax |